# Module 3 -- Verify MCP Gateway & Registry

Module 3 deploys a fully functional MCP Gateway and Registry backed by NGINX,
CloudFront, DocumentDB, and Cognito. This notebook verifies that the
infrastructure is operational by exercising the key integration points:

1. Resolve CloudFormation stack outputs (Cognito, Registry URL, CloudFront)
2. Test the Registry health endpoint
3. Authenticate using the M2M static token from Secrets Manager
4. List registered tools and agents
5. Verify M2M credentials in Secrets Manager
6. Test the Cognito OAuth2 client credentials flow

> **Demo logging:** some cells below print truncated token/credential prefixes to
> confirm authentication works in this ephemeral sandbox. Production systems should
> log metadata only — never credentials or full model responses.

In [ ]:
!pip install "boto3==1.42.87" "httpx==0.27.0" -q

## Step 1: Resolve CloudFormation Stack Outputs

Module 3 is deployed as nested CloudFormation stacks under `workshop-registry-stack`.
The key exports include the Registry URL, Cognito User Pool, and CloudFront distribution.

In [ ]:
import boto3, json

region = boto3.session.Session().region_name or "us-west-2"
cfn = boto3.client("cloudformation", region_name=region)

# Fetch all CFN exports
exports = {}
for page in cfn.get_paginator("list_exports").paginate():
    for e in page["Exports"]:
        exports[e["Name"]] = e["Value"]

# Key Module 3 exports
keys_of_interest = [
    "workshop-RegistryUrl",
    "workshop-MainCloudFrontUrl",
    "workshop-CognitoUserPoolId",
    "workshop-CognitoM2MClientId",
    "workshop-CognitoDomain",
]

print("Module 3 CloudFormation Exports")
print("=" * 60)
missing = []
for key in keys_of_interest:
    value = exports.get(key, "(not found)")
    print(f"  {key:40s} = {value}")
    if value == "(not found)":
        missing.append(key)

if missing:
    raise RuntimeError(
        f"Missing CloudFormation exports: {missing}. "
        "Is workshop-registry-stack deployed and CREATE_COMPLETE?"
    )

REGISTRY_URL = exports["workshop-RegistryUrl"]
CLOUDFRONT_URL = exports["workshop-MainCloudFrontUrl"]
COGNITO_POOL_ID = exports["workshop-CognitoUserPoolId"]
M2M_CLIENT_ID = exports["workshop-CognitoM2MClientId"]
COGNITO_DOMAIN = exports["workshop-CognitoDomain"]

## Step 2: Test Registry Health

The Registry API exposes a health endpoint through CloudFront and NGINX.

In [ ]:
import httpx

print(f"Testing health: {REGISTRY_URL}/health")
resp = httpx.get(f"{REGISTRY_URL}/health", timeout=15)
print(f"Status: {resp.status_code}")
print(f"Body:   {resp.text[:200]}")
resp.raise_for_status()

# Also test the CloudFront distribution directly
print(f"\nTesting CloudFront: {CLOUDFRONT_URL}/health")
resp2 = httpx.get(f"{CLOUDFRONT_URL}/health", timeout=15)
print(f"Status: {resp2.status_code}")
print(f"Body:   {resp2.text[:200]}")
resp2.raise_for_status()

## Step 3: Get Registry API Token from Secrets Manager

The Registry API requires a bearer token for authentication. This token is stored
in a Secrets Manager secret whose ARN is exported by the Registry CloudFormation
stack as `workshop-RegistryApiTokenSecretArn`.

In [ ]:
sm = boto3.client("secretsmanager", region_name=region)

# Get the registry API token (for authenticating to the Registry REST API)
api_token_arn = exports.get("workshop-RegistryApiTokenSecretArn", "")
if not api_token_arn:
    raise RuntimeError(
        "Export 'workshop-RegistryApiTokenSecretArn' not found. "
        "Is workshop-registry-stack deployed and CREATE_COMPLETE?"
    )

api_secret = json.loads(
    sm.get_secret_value(SecretId=api_token_arn)["SecretString"]
)

print(f"Secret ARN: {api_token_arn}")
print(f"Keys:       {list(api_secret.keys())}")
print(f"API Token:  {api_secret.get('api_token', 'N/A')[:20]}...")

ACCESS_TOKEN = api_secret.get("api_token", "")
AUTH_HEADERS = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

## Step 4: List Registered Tools and Agents

The Registry API provides endpoints for listing MCP servers (tools) and agents.
We authenticate using the static API token.

In [ ]:
# List all MCP servers (tools)
print("=== Registered MCP Servers (Tools) ===")
resp = httpx.get(f"{REGISTRY_URL}/api/servers", headers=AUTH_HEADERS, timeout=15)
resp.raise_for_status()
servers = resp.json()
if isinstance(servers, dict):
    servers = servers.get("servers", servers.get("items", []))

print(f"Total servers: {len(servers)}")
print("-" * 80)
MAX_PRINT = 20
for s in servers[:MAX_PRINT]:
    name = s.get("display_name") or s.get("server_name") or s.get("name", "?")
    desc = s.get("description", "")[:50]
    proxy = s.get("proxy_pass_url", "?")
    tags = s.get("tags", "")
    status = s.get("status", "?")
    print(f"  {name:30s}  status={status}")
    print(f"    proxy: {proxy}")
    print(f"    tags:  {tags}")
    if desc:
        print(f"    desc:  {desc}")
    print()
if len(servers) > MAX_PRINT:
    print(f"... and {len(servers) - MAX_PRINT} more")

In [ ]:
# List registered agents
print("=== Registered Agents ===")
resp = httpx.get(f"{REGISTRY_URL}/api/agents", headers=AUTH_HEADERS, timeout=15)
if resp.status_code == 200:
    agents = resp.json()
    if isinstance(agents, dict):
        agents = agents.get("agents", agents.get("items", []))
    print(f"Total agents: {len(agents)}")
    for a in agents:
        name = a.get("name", "?")
        desc = a.get("description", "")[:60]
        print(f"  {name}: {desc}")
else:
    raise RuntimeError(
        f"Registry agents endpoint returned {resp.status_code}: {resp.text[:300]}"
    )

## Step 5: Test Cognito OAuth2 Client Credentials Flow

Module 3's Cognito User Pool supports M2M authentication via the OAuth2
client_credentials grant. This is how service accounts authenticate
programmatically.

The M2M client ID is available from the CloudFormation export, and the client
secret is stored in a separate Secrets Manager secret created by the Registry
stack's Cognito initialization Lambda.

In [ ]:
import base64

# Read M2M client ID from CFN export (already resolved above)
m2m_client_id = M2M_CLIENT_ID

# Read M2M client secret from the Cognito M2M secret in Secrets Manager
m2m_secret_arn = exports.get("workshop-CognitoM2MClientSecretArn", "")
if not m2m_secret_arn:
    raise RuntimeError("Export 'workshop-CognitoM2MClientSecretArn' not found")

m2m_secret = json.loads(
    sm.get_secret_value(SecretId=m2m_secret_arn)["SecretString"]
)
m2m_client_secret = m2m_secret["client_secret"]

print(f"M2M Client ID:     {m2m_client_id[:15]}...")
print(f"M2M Client Secret: ***")

auth_str = base64.b64encode(
    f"{m2m_client_id}:{m2m_client_secret}".encode()
).decode()

# Build the Cognito token URL from the domain
cognito_token_url = f"https://{COGNITO_DOMAIN}.auth.{region}.amazoncognito.com/oauth2/token"
print(f"Token URL: {cognito_token_url}")

resp = httpx.post(
    cognito_token_url,
    headers={
        "Authorization": f"Basic {auth_str}",
        "Content-Type": "application/x-www-form-urlencoded",
    },
    data={"grant_type": "client_credentials", "scope": "mcp-servers-unrestricted/read mcp-servers-unrestricted/execute"},
    timeout=10,
)

print(f"Status: {resp.status_code}")
if resp.status_code == 200:
    token_data = resp.json()
    print(f"Token type:  {token_data.get('token_type')}")
    print(f"Expires in:  {token_data.get('expires_in')}s")
    print(f"Access token: {token_data['access_token'][:30]}...")
    print("\nCognito M2M auth is working!")
else:
    print(f"Error: {resp.text[:200]}")

## Step 6: Inspect Cognito User Pool

The Cognito User Pool is the canonical identity source for the entire platform.
Let's inspect its configuration.

In [ ]:
cognito = boto3.client("cognito-idp", region_name=region)

pool = cognito.describe_user_pool(UserPoolId=COGNITO_POOL_ID)["UserPool"]
print(f"User Pool: {pool['Name']} ({COGNITO_POOL_ID})")
print(f"Status:    {pool.get('Status', 'N/A')}")
print(f"Created:   {pool.get('CreationDate', 'N/A')}")

# List app clients
clients = []
paginator = cognito.get_paginator("list_user_pool_clients")
for page in paginator.paginate(UserPoolId=COGNITO_POOL_ID):
    clients.extend(page.get("UserPoolClients", []))
print(f"\nApp Clients ({len(clients)}):")
for c in clients:
    print(f"  {c['ClientName']:30s}  ID: {c['ClientId']}")

# List resource servers
rs = cognito.list_resource_servers(
    UserPoolId=COGNITO_POOL_ID, MaxResults=10
)["ResourceServers"]
print(f"\nResource Servers ({len(rs)}):")
for server in rs:
    print(f"  {server['Identifier']}")
    for scope in server.get("Scopes", []):
        print(f"    Scope: {scope['ScopeName']} - {scope.get('ScopeDescription', '')}")

## Summary

Module 3 is verified and operational:

| Component | Status |
|---|---|
| CloudFront distribution | Serving traffic |
| Registry API | Healthy and responding |
| MCP Servers | Registered and discoverable |
| Cognito User Pool | Active with M2M support |
| Secrets Manager | M2M credentials stored |
| OAuth2 flow | Client credentials working |

**Next:** Proceed to Module 4 where you will add a governed access path
through the AgentCore Gateway.